# 30 — Experimento B: corrida formal con las cuatro condiciones

**Autor:** Jason Sepúlveda S.
**Proyecto:** Evaluación experimental de la extracción automática y la representación
persistente de decisiones organizacionales frente a Recuperación Aumentada (RAG)
**Depende de:** [ADR 0009](../docs/decisions/0009-adr-regla-de-citacion-correcta.md) ·
[ADR 0010](../docs/decisions/0010-adr-decisiones-del-analisis-del-experimento-b.md) ·
[ADR 0014](../docs/decisions/0014-adr-c1-como-rag-convencional.md) ·
[ADR 0017](../docs/decisions/0017-adr-medicion-de-h4.md) ·
[ADR 0018](../docs/decisions/0018-adr-medicion-de-h2-y-h3.md) ·
[ADR 0019](../docs/decisions/0019-adr-c0-en-la-corrida-formal-del-experimento-b.md) ·
[ADR 0024](../docs/decisions/0024-adr-hoja-oe4-autocontenida.md) ·
[notebook 28](28-jss-experimento-a.ipynb) · issue #294 (épica #288)

**Alcance.** Registro de análisis de la corrida formal `experimento-b-formal`: procedencia, resultados
principales por condición y estrato, validez del juez automático frente a la verificación humana,
etapa del fallo de C2 (OE4), citas y abstención, y lectura con sus límites. El notebook solo lee las
salidas guardadas en `reports/experiment-b/experimento-b-formal/`; no llama a ningún modelo ni
recalcula la corrida. **Requisito:** ninguna sección necesita Ollama ni el corpus; la sección 1
muestra además, si existen en el equipo, algunos archivos locales de `data/interim/` (se omiten sin
error cuando faltan).

> **Aviso de alcance (ADR 0005).** Las series son series de **evaluación**. El notebook muestra solo
> **identificadores de pregunta y de serie, etiquetas y cifras agregadas**. No muestra ni cita texto de
> respuestas, de transcripciones ni de decisiones.

## Los términos, en simple

| Término | Qué significa |
|---|---|
| Condición | Qué contexto recibe el mismo generador para responder: C0 ninguno, C1 fragmentos del texto, C2 la base compilada automáticamente, C3 la base con las decisiones de la referencia humana. |
| Estrato | Tipo de pregunta: E1 hecho puntual de una decisión, E2 estado vigente, E3 evolución (relaciones entre decisiones), E4 sin respuesta en el corpus. |
| Juez | Modelo que decide si una respuesta es correcta (`correct`) y si su evidencia la respalda (`evidence_supported`). |
| Kappa | Acuerdo entre el juez y las personas descontando el azar; el umbral preregistrado es 0,70. |
| Diferencia pareada | Diferencia de la tasa de corrección entre dos condiciones, pregunta por pregunta, con intervalo bootstrap del 95 %. |
| Abstención | La condición responde que no encuentra la respuesta en su contexto. |

---

# 1. La corrida y su procedencia

**Qué se hace.** Se lee el manifiesto de la corrida y se muestran los datos que permiten
reproducirla: condiciones, preguntas, repeticiones, registros y errores, commit, estado del árbol
de trabajo, entorno fijado, huellas de modelo y versión de OpenKOS. Después se explica qué recibe
cada condición.

**Dónde está el informe.** El issue #294 menciona un `informe.json`; ese nombre es el del
Experimento A. El comando `afg experiment b-report` del Experimento B guarda su informe en
`analisis.json` (o en `analisis-descriptivo.json` si la corrida no es reportable), y es el archivo que
se lee aquí. Los registros están en `respuestas.jsonl` y las hojas de verificación en
`muestra-juez*.csv` y `oe4-etapa-e1.csv`.

In [1]:
import json
import warnings

import pandas as pd
from IPython.display import Markdown, display

from afg.evaluation.judge_human_check import (
    check_judge_against_humans,
    check_judge_by_condition,
    read_csv_rows,
)
from afg.experiments.final_tables import b_cells_table, b_paired_table
from afg.experiments.oe4_summary import summarize_oe4_sheet
from afg.experiments.runner import load_records
from afg.experiments.summary import stratum_summary
from afg.shared.paths import EXPERIMENT_B_DIR, PROJECT_ROOT

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 90)

RUN = "experimento-b-formal"
RUN_DIR = EXPERIMENT_B_DIR / RUN
CONDITIONS = ("C0", "C1", "C2", "C3")
STRATA = ("E1", "E2", "E3", "E4", "ALL")


def read(name):
    return json.loads((RUN_DIR / name).read_text(encoding="utf-8"))


def es(value, digits=2):
    return f"{value:.{digits}f}".replace(".", ",")


def pct(value, digits=0):
    return es(100 * value, digits) + " %"


def ci(estimate, digits=2):
    if estimate is None:
        return "—"
    point = es(estimate["point"], digits)
    lower = es(estimate["lower"], digits)
    upper = es(estimate["upper"], digits)
    return f"{point} [{lower}; {upper}]"


manifest = read("manifest.json")
analysis = read("analisis.json")
juez = read("juez-humano.json")
oe4 = read("oe4-resumen.json")
records = load_records(RUN_DIR / "respuestas.jsonl", latest_only=True)

In [2]:
counts = manifest["counts"]
pinned = manifest["pinned_environment"]
pd.Series(
    {
        "corrida": manifest["run_name"],
        "partición": manifest["partition"],
        "condiciones": ", ".join(manifest["conditions"]),
        "series": f"{len(manifest['series'])} ({', '.join(manifest['series'])})",
        "preguntas": counts["questions"],
        "repeticiones por pregunta": manifest["settings"]["runs"],
        "registros esperados / completos": f"{counts['expected']} / {counts['complete']}",
        "registros con error": counts["errors"],
        "commit de la corrida": manifest["git_commit"][:12],
        "git_dirty": manifest["git_dirty"],
        "congelamiento": (
            f"ADR {manifest['freeze']['adr']}, commit {manifest['freeze']['code_commit'][:12]}"
        ),
        "entorno fijado cumplido": manifest["pinned_environment_ok"],
        "generador": (
            f"{manifest['settings']['generative_model']} (temperatura "
            f"{manifest['settings']['temperature']}, semilla {manifest['settings']['base_seed']})"
        ),
        "juez": (
            f"{manifest['settings']['judge_model']} (orden {manifest['settings']['judge_order']}, "
            f"temperatura {manifest['settings']['judge_temperature']})"
        ),
        "modelo de suficiencia": manifest["sufficiency_model"],
        "Ollama": pinned["ollama_version"],
        "OpenKOS": pinned["openkos_version"],
        "C1 (fragmentos)": (
            f"tamaño {manifest['c1_params']['chunk_size']}, solape "
            f"{manifest['c1_params']['chunk_overlap']}, k = {manifest['c1_params']['k']}"
        ),
        "extracción de C2": manifest["c2_extraction"],
        "espacios de C3": manifest["c3_workspaces"],
    },
    name="valor",
).to_frame()

,valor
corrida,experimento-b-formal
partición,evaluation
condiciones,"C0, C1, C2, C3"
series,"11 (ES2002, ES2008, ES2014, ES2016, IS1003, IS1006, IS1008, IS1009, TS3003, TS3005, TS..."
preguntas,99
repeticiones por pregunta,3
registros esperados / completos,1188 / 1188
registros con error,0
commit de la corrida,c6ee4500b9c7
git_dirty,False


In [3]:
pd.DataFrame(
    [
        {"modelo": name, "huella (sha256)": digest[:16] + "…"}
        for name, digest in pinned["model_digests"].items()
    ]
)

,modelo,huella (sha256)
0,qwen3:8b,500a1f067a9f7826…
1,gpt-oss:20b,f38aa0c53da5f8c4…
2,gemma4:26b-a4b,001e5dafc3c77684…
3,bge-m3,7907646426070047…


**Cómo responde cada condición.** Las cuatro condiciones reciben la misma pregunta y usan
`qwen3:8b` para redactar la respuesta. C1 y C2 también difieren en la etapa de respuesta.

| Condición | Contexto y procedimiento de respuesta | Origen |
|---|---|---|
| C0 | Responde sin recuperar contexto. | Referencia descriptiva (ADR 0019). |
| C1 | Recupera hasta cinco fragmentos de la transcripción y responde en una llamada, con las instrucciones del RAG convencional. | Búsqueda híbrida sobre la transcripción (ADR 0014). |
| C2 | Recupera objetos de la base compilada y comprueba si el contexto alcanza antes de responder. | Flujo de consulta de OpenKOS sobre la base del Experimento A. |
| C3 | Usa la consulta de C2 sobre una copia de su base con las decisiones y sus enlaces sustituidos por los de la referencia humana. | La composición queda registrada en `c3-manifest.json` (ADR 0013). |

La comparación C1–C2 mide el desempeño de los dos sistemas completos. Para estimar por separado
el aporte de la representación haría falta usar el mismo procedimiento de respuesta en ambos.
En C2–C3 la consulta es la misma y se sustituyen las decisiones con sus enlaces. Los conceptos,
eventos, proyectos y fuentes siguen siendo los de la extracción automática.

**Un ejemplo concreto** (serie ES2002, repetición 0). Las rutas de `data/interim/` no se versionan.
Las siguientes celdas muestran estos ejemplos si los archivos están disponibles localmente.

In [4]:
EXAMPLE_SERIES, EXAMPLE_RUN = "ES2002", 0
workspace = f"{EXAMPLE_SERIES}/run-{EXAMPLE_RUN}"
c2_dir = PROJECT_ROOT / "data" / "interim" / "openkos" / manifest["c2_extraction"] / workspace
c3_dir = PROJECT_ROOT / manifest["c3_workspaces"] / manifest["c2_extraction"] / workspace
transcript = PROJECT_ROOT / "data" / "interim" / "transcripts" / f"{EXAMPLE_SERIES}a.md"
paths = [
    ("C1", "transcripción cruda, en fragmentos", transcript),
    ("C2", "bundle de la extracción automática", c2_dir / "bundle"),
    ("C3", "bundle con decisiones de la referencia", c3_dir / "bundle"),
]
pd.DataFrame(
    [
        {
            "condición": condition,
            "contenido": what,
            "ruta": str(path.relative_to(PROJECT_ROOT)),
            "disponible en este equipo": "sí" if path.exists() else "no",
        }
        for condition, what, path in paths
    ]
)

,condición,contenido,ruta,disponible en este equipo
0,C1,"transcripción cruda, en fragmentos",data/interim/transcripts/ES2002a.md,sí
1,C2,bundle de la extracción automática,data/interim/openkos/experimento-a-formal/ES2002/run-0/bundle,sí
2,C3,bundle con decisiones de la referencia,data/interim/openkos-c3/experimento-a-formal/ES2002/run-0/bundle,sí


In [5]:
NAMESPACES = ("decisions", "concepts", "events", "projects", "sources")
if (c2_dir / "bundle").exists() and (c3_dir / "bundle").exists():
    display(
        pd.DataFrame(
            {
                ns: {
                    "C2": len(list((c2_dir / "bundle" / ns).glob("*.md"))),
                    "C3": len(list((c3_dir / "bundle" / ns).glob("*.md"))),
                }
                for ns in NAMESPACES
            }
        ).rename_axis("objetos por carpeta")
    )
else:
    print("Los espacios de trabajo de ejemplo no están en este equipo: se omite el conteo.")

c3_manifest_path = c3_dir / "c3-manifest.json"
if c3_manifest_path.exists():
    c3_manifest = json.loads(c3_manifest_path.read_text(encoding="utf-8"))
    print("Trazabilidad de C3 (c3-manifest.json):")
    for key in ("series_id", "c2_run", "run", "generated_by", "openkos_version"):
        print(f"  {key}: {c3_manifest[key]}")
    print(f"  decisiones en la referencia: {c3_manifest['reference_decisions']}")
    print(f"  decisiones escritas: {c3_manifest['decisions_written']}")
    print(f"  decisiones descartadas: {len(c3_manifest['dropped'])}")
    print(f"  relaciones escritas: {c3_manifest['relations_written']}")
    print(f"  relaciones descartadas: {len(c3_manifest['relations_dropped'])}")
else:
    print("c3-manifest.json no está en este equipo: se omite la trazabilidad de C3.")

,decisions,concepts,events,projects,sources
objetos por carpeta,,,,,
C2,54,19,2,1,4
C3,29,19,2,1,4


Trazabilidad de C3 (c3-manifest.json):
  series_id: ES2002
  c2_run: experimento-a-formal
  run: 0
  generated_by: afg/c3-gold
  openkos_version: 0.7.0
  decisiones en la referencia: 31
  decisiones escritas: 29
  decisiones descartadas: 2
  relaciones escritas: 0
  relaciones descartadas: 3


In [6]:
display(
    Markdown(
        f"**Lectura.** La corrida cubre {counts['questions']} preguntas de "
        f"{len(manifest['series'])} series de "
        f"evaluación con {len(manifest['conditions'])} condiciones y "
        f"{manifest['settings']['runs']} repeticiones: "
        f"{counts['records']} registros, {counts['complete']} completos y {counts['errors']} "
        f"con error. Se ejecutó "
        f"con el commit `{manifest['git_commit'][:12]}`, árbol de trabajo limpio "
        f"(`git_dirty = {manifest['git_dirty']}`), bajo el congelamiento del ADR "
        f"{manifest['freeze']['adr']} y con el entorno fijado cumplido "
        f"(`{manifest['pinned_environment_ok']}`: Ollama {pinned['ollama_version']}, OpenKOS "
        f"{pinned['openkos_version']} y las huellas de modelo de la tabla). El informe es "
        f"{'reportable' if analysis['reportable'] else 'no reportable'}."
    )
)

**Lectura.** La corrida cubre 99 preguntas de 11 series de evaluación con 4 condiciones y 3 repeticiones: 1188 registros, 1188 completos y 0 con error. Se ejecutó con el commit `c6ee4500b9c7`, árbol de trabajo limpio (`git_dirty = False`), bajo el congelamiento del ADR 0025 y con el entorno fijado cumplido (`True`: Ollama 0.32.9, OpenKOS 0.7.0 y las huellas de modelo de la tabla). El informe es reportable.

---

# 2. Resultados principales

**Pregunta.** ¿Qué parte de las preguntas contesta correctamente cada condición, por estrato, y cómo
se comparan C2 y C1 (criterio D16), C3 y C2 (H2) y C1 y C2 frente a C0?

**Qué se muestra.** La tasa de corrección (`correct`, juzgada por el modelo) por condición y estrato
con su intervalo bootstrap del 95 %. Las corridas de una misma pregunta se promedian por pregunta y el
intervalo remuestrea preguntas, no respuestas. Las cifras salen de `analisis.json`.

In [7]:
cells = b_cells_table([(RUN, analysis)])
cells = cells[cells["condition"].isin(CONDITIONS)]
questions_per_stratum = cells[cells["condition"] == "C0"].set_index("stratum")["questions"]
display(
    pd.DataFrame(
        {"preguntas": [int(questions_per_stratum[s]) for s in STRATA]},
        index=pd.Index(STRATA, name="estrato"),
    ).T
)
table = {
    s: {
        c: ci(
            {
                "point": row["correct"],
                "lower": row["correct_lower"],
                "upper": row["correct_upper"],
            }
        )
        for c, row in cells[cells["stratum"] == s].set_index("condition").iterrows()
    }
    for s in STRATA
}
pd.DataFrame(table).loc[list(CONDITIONS), list(STRATA)].rename_axis("corrección [IC 95 %]")

estrato,E1,E2,E3,E4,ALL
preguntas,25,25,24,25,99


,E1,E2,E3,E4,ALL
corrección [IC 95 %],,,,,
C0,"0,17 [0,04; 0,33]","0,04 [0,00; 0,12]","0,01 [0,00; 0,04]","0,40 [0,21; 0,59]","0,16 [0,09; 0,23]"
C1,"0,72 [0,55; 0,88]","0,64 [0,44; 0,84]","0,54 [0,36; 0,72]","0,96 [0,88; 1,00]","0,72 [0,63; 0,80]"
C2,"0,43 [0,25; 0,60]","0,35 [0,19; 0,52]","0,17 [0,04; 0,32]","1,00 [1,00; 1,00]","0,49 [0,39; 0,58]"
C3,"0,72 [0,56; 0,88]","0,76 [0,60; 0,91]","0,71 [0,54; 0,86]","1,00 [1,00; 1,00]","0,80 [0,72; 0,87]"


In [8]:
failures = cells[["condition", "stratum", "judge_failures", "errors"]]
print(
    "Fallos del juez:",
    int(failures["judge_failures"].sum()),
    "| registros con error:",
    int(failures["errors"].sum()),
    "| celdas por debajo del mínimo preregistrado:",
    int(sum(c["below_minimum"] for c in analysis["cells"])),
)
print("Advertencias del análisis:", analysis["warnings"] or "ninguna")
display(Markdown("**Nota descriptiva del análisis (C0):** " + analysis["descriptive_note"]))
display(Markdown("**Criterio D16 tal como lo guarda el análisis:** " + analysis["criterion_d16"]))

Fallos del juez: 0 | registros con error: 0 | celdas por debajo del mínimo preregistrado: 0
Advertencias del análisis: ninguna


**Nota descriptiva del análisis (C0):** C0 (ADR 0019): el mismo generador sin ningún contexto recuperado, referencia sin recuperación. C1 - C0 se lee como el valor de recuperar texto y C2 - C0 como el de la base compilada frente a no recuperar nada; se calculan sobre la corrección, aparean por pregunta y son descriptivas: no deciden D16 ni H1-H4. C0 no cita ni aporta evidencia, así que sus métricas de evidencia y de citación no aplican y se informan como no disponibles; una respuesta de C0 que acierta no prueba conocimiento previo del corpus ni lo descarta.

**Criterio D16 tal como lo guarda el análisis:** Criterio D16 (enmendado, ADR 0010): la hipótesis se respalda si el intervalo bootstrap del 95 % de la diferencia pareada C2 - C1 queda por encima de 0; en otro caso el resultado es no concluyente. La no superposición de los intervalos independientes de C2 y C1 (redacción original) se informa aparte como robustez y no reemplaza al criterio principal. Desde el 2026-11-02 el veredicto exige el mínimo prerregistrado de preguntas juzgadas por condición en el estrato; por debajo se informa insufficient_sample (muestra insuficiente) con las cifras a la vista.

**Criterio D16 y comparaciones pareadas.** D16 se evalúa en E3 con la diferencia pareada C2 − C1
sobre la corrección. Se muestran además, de forma descriptiva, las demás comparaciones guardadas
sobre `correct`. En «C2 − C3» una diferencia negativa significa que C3 acierta más que C2; C0 es la
referencia de C1 − C0 y C2 − C0 y esas dos comparaciones no deciden D16 ni H1–H4.

In [9]:
paired = b_paired_table([(RUN, analysis)])
paired = paired[paired["metric"] == "correct"].copy()
paired["comparación"] = paired["condition_a"] + " - " + paired["condition_b"]
paired["diferencia [IC 95 %]"] = [
    f"{es(d, 2)} [{es(lo, 2)}; {es(hi, 2)}]"
    for d, lo, hi in zip(
        paired["difference"], paired["difference_lower"], paired["difference_upper"], strict=True
    )
]
paired["rol"] = [
    "D16 (primaria)" if p else "descriptiva" if d else "secundaria"
    for p, d in zip(paired["primary"], paired["descriptive"], strict=True)
]
paired = paired.rename(columns={"stratum": "estrato", "n_pairs": "pares", "criterion": "criterio"})
order = {c: i for i, c in enumerate(["C2 - C1", "C2 - C3", "C1 - C0", "C2 - C0"])}
paired["_o"] = paired["comparación"].map(order)
paired.sort_values(["_o", "estrato"])[
    ["comparación", "estrato", "pares", "diferencia [IC 95 %]", "criterio", "rol"]
].reset_index(drop=True)

,comparación,estrato,pares,diferencia [IC 95 %],criterio,rol
0,C2 - C1,ALL,99,"-0,23 [-0,33; -0,13]",inconclusive,secundaria
1,C2 - C1,E1,25,"-0,29 [-0,49; -0,08]",inconclusive,secundaria
2,C2 - C1,E2,25,"-0,29 [-0,51; -0,08]",inconclusive,secundaria
3,C2 - C1,E3,24,"-0,38 [-0,61; -0,14]",inconclusive,D16 (primaria)
4,C2 - C1,E4,25,"0,04 [0,00; 0,12]",inconclusive,secundaria
5,C2 - C3,ALL,99,"-0,31 [-0,40; -0,22]",inconclusive,secundaria
6,C2 - C3,E1,25,"-0,29 [-0,51; -0,07]",inconclusive,secundaria
7,C2 - C3,E2,25,"-0,41 [-0,60; -0,24]",inconclusive,secundaria
8,C2 - C3,E3,24,"-0,54 [-0,72; -0,36]",inconclusive,secundaria
9,C2 - C3,E4,25,"0,00 [0,00; 0,00]",inconclusive,secundaria


In [10]:
def difference(a, b, stratum):
    return next(
        c["difference"]
        for c in analysis["comparisons"]
        if (c["condition_a"], c["condition_b"], c["stratum"], c["metric"])
        == (a, b, stratum, "correct")
    )


d16 = next(c for c in analysis["comparisons"] if c["primary"])
c0_all = next(c for c in analysis["cells"] if c["condition"] == "C0" and c["stratum"] == "ALL")[
    "metrics"
]["correct"]
where = (
    "por completo por debajo de 0"
    if d16["difference_upper"] < 0
    else "en un rango que no está bajo 0"
)
(a_lo, a_hi), (b_lo, b_hi) = d16["interval_a"], d16["interval_b"]
below_zero = d16["difference_upper"] < 0
h2 = analysis["h2"]
c3_minus_c2 = {
    c["stratum"]: -c["difference"]
    for c in analysis["comparisons"]
    if (c["condition_a"], c["condition_b"], c["metric"]) == ("C2", "C3", "correct")
}
display(
    Markdown(
        f"**D16.** En {d16['stratum']} la diferencia pareada {d16['condition_a']} - "
        f"{d16['condition_b']} es "
        f"{es(d16['difference'])} con intervalo del 95 % [{es(d16['difference_lower'])}; "
        f"{es(d16['difference_upper'])}] sobre {d16['n_pairs']} preguntas (muestra suficiente: "
        f"{d16['sample_sufficient']}). El criterio exige que el intervalo quede **por encima** "
        f"de 0 y, en "
        f"cualquier otro caso, la herramienta imprime «{d16['criterion']}». Aquí el intervalo "
        f"queda "
        f"{where}: la hipótesis "
        f"de que C2 supera a C1 en E3 **no se respalda**, y la dirección observada es la "
        f"contraria. La "
        f"etiqueta «{d16['criterion']}» no distingue «sin evidencia» de «evidencia en contra»; "
        f"hay que leer el intervalo. Los intervalos independientes de cada condición "
        f"{'no se superponen' if not d16['intervals_overlap'] else 'se superponen'} "
        f"(C2 {es(a_lo)} a {es(a_hi)}; C1 {es(b_lo)} a {es(b_hi)}), y la redacción original "
        f"por no superposición también da "
        f"«{d16['nonoverlap_criterion']}».\n\n"
        f"**H2.** La brecha de corrección C3 - C2 es {es(h2['gap_e3'])} en E3 y "
        f"{es(h2['gap_e1'])} en E1; su "
        f"diferencia es {es(h2['difference'])} [{es(h2['difference_lower'])}; "
        f"{es(h2['difference_upper'])}] "
        f"({h2['n_e3']} preguntas de E3 y {h2['n_e1']} de E1), y el veredicto es "
        f"«{h2['verdict']}»: el "
        f"intervalo incluye 0 y no permite afirmar que la brecha sea mayor en E3 que en E1.\n\n"
        f"**C0 como referencia.** Sin contexto, C0 acierta {ci(c0_all)} "
        f"en el total. Frente a C0, C1 gana {es(difference('C1', 'C0', 'ALL'))} "
        f"y C2 gana {es(difference('C2', 'C0', 'ALL'))} "
        f"en el total; son comparaciones descriptivas. Una respuesta acertada de C0 no prueba "
        f"conocimiento "
        f"previo del corpus ni lo descarta."
    )
)

**D16.** En E3 la diferencia pareada C2 - C1 es -0,38 con intervalo del 95 % [-0,61; -0,14] sobre 24 preguntas (muestra suficiente: True). El criterio exige que el intervalo quede **por encima** de 0 y, en cualquier otro caso, la herramienta imprime «inconclusive». Aquí el intervalo queda por completo por debajo de 0: la hipótesis de que C2 supera a C1 en E3 **no se respalda**, y la dirección observada es la contraria. La etiqueta «inconclusive» no distingue «sin evidencia» de «evidencia en contra»; hay que leer el intervalo. Los intervalos independientes de cada condición no se superponen (C2 0,04 a 0,32; C1 0,36 a 0,72), y la redacción original por no superposición también da «inconclusive».

**H2.** La brecha de corrección C3 - C2 es 0,54 en E3 y 0,29 en E1; su diferencia es 0,25 [-0,04; 0,54] (24 preguntas de E3 y 25 de E1), y el veredicto es «inconclusive»: el intervalo incluye 0 y no permite afirmar que la brecha sea mayor en E3 que en E1.

**C0 como referencia.** Sin contexto, C0 acierta 0,16 [0,09; 0,23] en el total. Frente a C0, C1 gana 0,56 y C2 gana 0,33 en el total; son comparaciones descriptivas. Una respuesta acertada de C0 no prueba conocimiento previo del corpus ni lo descarta.

---

# 3. Validez del juez automático

**Pregunta.** ¿Concuerda el juez con una persona lo bastante como para fiarse de sus etiquetas?

**Qué se muestra.** El acuerdo entre el juez y la etiqueta humana en la muestra de verificación
(`muestra-juez.csv`, con las etiquetas de la persona, y `muestra-juez.clave.csv`, con la condición y
las etiquetas del juez, unidas por `id`). El kappa de Cohen se compara con el umbral preregistrado de
0,70 por etiqueta; `correct` decide si se mantiene el juicio automático, y `evidence_supported` y
`outdated` tienen su propia bandera de confiabilidad.

In [11]:
rows = []
for column in juez["columns"] + [juez["outdated"]]:
    kappa = column["kappa"]
    rows.append(
        {
            "etiqueta": column["column"],
            "ítems": column["n_items"],
            "acuerdo observado": es(column["observed_agreement"], 3),
            "positivos del juez": pct(column["judge_positive_rate"], 1),
            "positivos humanos": pct(column["human_positive_rate"], 1),
            "kappa": "indefinido" if kappa is None else es(kappa, 3),
            f"cumple {es(juez['min_kappa'])}": "sí" if column["passes"] else "no",
        }
    )
pd.DataFrame(rows)

,etiqueta,ítems,acuerdo observado,positivos del juez,positivos humanos,kappa,"cumple 0,70"
0,correct,240,"0,904","52,9 %","56,7 %","0,807",sí
1,evidence_supported,240,"0,754","60,4 %","80,0 %","0,438",no
2,outdated,42,"1,000","0,0 %","0,0 %",indefinido,no


In [12]:
key_rows = read_csv_rows(RUN_DIR / "muestra-juez.clave.csv")
sheet_rows = read_csv_rows(RUN_DIR / "muestra-juez.csv")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    recomputed = check_judge_against_humans(key_rows, sheet_rows, min_kappa=juez["min_kappa"])
    by_condition = check_judge_by_condition(key_rows, sheet_rows, min_kappa=juez["min_kappa"])
stored = {c["column"]: c["kappa"] for c in juez["columns"]}
assert all(abs(c.kappa - stored[c.column]) < 1e-9 for c in recomputed.columns)
print("El kappa recalculado desde las dos hojas coincide con juez-humano.json.")

per_condition = []
for condition, check in by_condition.items():
    for column in check.columns:
        per_condition.append(
            {
                "condición": condition,
                "etiqueta": column.column,
                "ítems": column.n_items,
                "acuerdo observado": es(column.observed_agreement, 3),
                "positivos del juez": pct(column.judge_positive_rate, 1),
                "positivos humanos": pct(column.human_positive_rate, 1),
                "kappa": "indefinido" if column.kappa is None else es(column.kappa, 3),
                f"cumple {es(juez['min_kappa'])}": "sí" if column.passes else "no",
            }
        )
per_condition = pd.DataFrame(per_condition)
per_condition[per_condition["etiqueta"] == "correct"].reset_index(drop=True)

El kappa recalculado desde las dos hojas coincide con juez-humano.json.


,condición,etiqueta,ítems,acuerdo observado,positivos del juez,positivos humanos,kappa,"cumple 0,70"
0,C0,correct,60,"0,817","16,7 %","28,3 %","0,484",no
1,C1,correct,60,"0,883","70,0 %","71,7 %","0,718",sí
2,C2,correct,60,"0,950","45,0 %","46,7 %","0,899",sí
3,C3,correct,60,"0,967","80,0 %","80,0 %","0,896",sí


In [13]:
per_condition[per_condition["etiqueta"] == "evidence_supported"].reset_index(drop=True)

,condición,etiqueta,ítems,acuerdo observado,positivos del juez,positivos humanos,kappa,"cumple 0,70"
0,C0,evidence_supported,60,"0,383","20,0 %","78,3 %","0,080",no
1,C1,evidence_supported,60,"0,933","61,7 %","61,7 %","0,859",sí
2,C2,evidence_supported,60,"0,867","76,7 %","90,0 %","0,535",no
3,C3,evidence_supported,60,"0,833","83,3 %","90,0 %","0,286",no


In [14]:
correct = next(c for c in juez["columns"] if c["column"] == "correct")
evidence = next(c for c in juez["columns"] if c["column"] == "evidence_supported")
outdated = juez["outdated"]
kappa_correct = {cond: check.columns[0] for cond, check in by_condition.items()}
below = [c for c, v in kappa_correct.items() if not v.passes]
threshold = es(juez["min_kappa"])
below_text = (
    f"Quedan por debajo de {threshold}: {', '.join(below)}."
    if below
    else "Todas cumplen el umbral."
)
display(
    Markdown(
        f"**Lectura.** *correct*: kappa {es(correct['kappa'], 3)} sobre {correct['n_items']} ítems "
        f"(acuerdo {pct(correct['observed_agreement'], 1)}), por encima del umbral de "
        f"{es(juez['min_kappa'])}; "
        f"se mantiene el juicio automático de la corrección. *evidence_supported*: kappa "
        f"{es(evidence['kappa'], 3)}, **no cumple**; el juez es más estricto que las personas "
        f"(marca "
        f"«sí» en {pct(evidence['judge_positive_rate'], 1)} de los ítems frente a "
        f"{pct(evidence['human_positive_rate'], 1)} de las personas), de modo que esa métrica "
        f"no es "
        f"confiable y solo debe leerse como descriptiva o con evaluación humana. *outdated*: "
        f"en los "
        f"{outdated['n_items']} ítems que el juez evaluó, ni el juez ni las personas marcaron «sí» "
        f"(positivos {pct(outdated['judge_positive_rate'], 1)} y "
        f"{pct(outdated['human_positive_rate'], 1)}); "
        f"con una sola clase el kappa es indefinido y cuenta como «no cumple». **H4 no es "
        f"evaluable con el "
        f"juez**: sin ninguna marca positiva no hay tasa de respuestas desactualizadas que "
        f"comparar, "
        f"aunque el acuerdo observado sea {pct(outdated['observed_agreement'], 0)}.\n\n"
        f"**Por condición (exploratorio).** Con {kappa_correct['C0'].n_items} ítems por "
        f"condición, el kappa "
        f"de *correct* es "
        + ", ".join(f"{c} {es(v.kappa, 2)}" for c, v in kappa_correct.items())
        + f". {below_text} "
        f"En C0, el juez marca «sí» en {pct(kappa_correct['C0'].judge_positive_rate, 0)} de la "
        f"muestra y las "
        f"personas en {pct(kappa_correct['C0'].human_positive_rate, 0)}: el juez es más "
        f"estricto, de modo que la "
        f"tasa de C0 podría estar subestimada y las diferencias C1 - C0 y C2 - C0, "
        f"sobreestimadas. El umbral "
        f"preregistrado se aplica a la muestra completa, no a cada condición, y 60 ítems por "
        f"condición dan "
        f"estimaciones inestables; estas cifras son una descripción, no un contraste."
    )
)

**Lectura.** *correct*: kappa 0,807 sobre 240 ítems (acuerdo 90,4 %), por encima del umbral de 0,70; se mantiene el juicio automático de la corrección. *evidence_supported*: kappa 0,438, **no cumple**; el juez es más estricto que las personas (marca «sí» en 60,4 % de los ítems frente a 80,0 % de las personas), de modo que esa métrica no es confiable y solo debe leerse como descriptiva o con evaluación humana. *outdated*: en los 42 ítems que el juez evaluó, ni el juez ni las personas marcaron «sí» (positivos 0,0 % y 0,0 %); con una sola clase el kappa es indefinido y cuenta como «no cumple». **H4 no es evaluable con el juez**: sin ninguna marca positiva no hay tasa de respuestas desactualizadas que comparar, aunque el acuerdo observado sea 100 %.

**Por condición (exploratorio).** Con 60 ítems por condición, el kappa de *correct* es C0 0,48, C1 0,72, C2 0,90, C3 0,90. Quedan por debajo de 0,70: C0. En C0, el juez marca «sí» en 17 % de la muestra y las personas en 28 %: el juez es más estricto, de modo que la tasa de C0 podría estar subestimada y las diferencias C1 - C0 y C2 - C0, sobreestimadas. El umbral preregistrado se aplica a la muestra completa, no a cada condición, y 60 ítems por condición dan estimaciones inestables; estas cifras son una descripción, no un contraste.

### Una respuesta que el juez y Germán calificaron distinto

Al revisar q015 y q019, encontramos cinco respuestas que el juez marcó como incorrectas y que
merecían una segunda lectura. Podemos comprobar una de ellas con la muestra de verificación:
en **q015, corrida 2**, el juez marcó «no» y Germán marcó «sí» en `correct`. La tabla cruza la
clave de la muestra y la hoja de verificación por `id` para mostrar ambos veredictos.

La corrida 0 de q015 y las tres de q019 quedaron fuera de la muestra. Sus etiquetas de OE4
describen la etapa del fallo atribuido, mientras que la columna de verificación indica si
tenemos un segundo juicio sobre la respuesta. La corrida 1 de q015 se incluye para mostrar que
el juez sí la consideró correcta. El análisis formal conserva el criterio de corrección fijado
para la corrida completa.

In [15]:
by_key = {(r["question_id"], r["condition"], r["run"]): r for r in records}
oe4_rows = read_csv_rows(RUN_DIR / "oe4-etapa-e1.csv")
stage_of = {(r["pregunta_id"], int(r["corrida"])): r["etapa_humana"] for r in oe4_rows}
human_correct = {r["id"]: r["correct"] for r in sheet_rows}
verified_correct = {
    (r["question_id"], r["condition"], int(r["run"])): human_correct[r["id"]] for r in key_rows
}
correct_label = {"si": "sí", "no": "no", None: "fuera de la muestra"}
pd.DataFrame(
    [
        {
            "pregunta": qid,
            "corrida": run,
            "correcta según el juez": "sí"
            if by_key[(qid, "C2", run)]["verdict"]["correct"]
            else "no",
            "correcta según Germán": correct_label[verified_correct.get((qid, "C2", run))],
            "etapa OE4": stage_of.get((qid, run), "—"),
        }
        for qid in ("q015", "q019")
        for run in range(3)
    ]
)

,pregunta,corrida,correcta según el juez,correcta según Germán,etapa OE4
0,q015,0,no,fuera de la muestra,sintesis
1,q015,1,sí,fuera de la muestra,—
2,q015,2,no,sí,sintesis
3,q019,0,no,fuera de la muestra,extraccion
4,q019,1,no,fuera de la muestra,extraccion
5,q019,2,no,fuera de la muestra,extraccion


---

# 4. OE4: en qué etapa falla C2 en E1

**Pregunta.** Cuando C2 responde mal en E1, ¿el dato se perdió al extraer, al recuperar o al sintetizar?

**Qué se muestra.** Primero la cifra principal, sobre **todas** las respuestas incorrectas de C2 en
E1 según los registros: las filas que una persona etiquetó a mano en `oe4-etapa-e1.csv` más las que la
hoja deja fuera por diseño porque la regla automática las atribuye sola a la síntesis
(`include_rule_attributed = False`). Después, en la subsección 4.1, el detalle de las filas etiquetadas
a mano, tal como lo guarda `oe4-resumen.json`.

In [16]:
from afg.experiments.c2_base_evidence import (
    count_stages,
    rule_attributed_failures,
    stage_shares,
)
from afg.experiments.c2_context import context_rows, error_evidence
from afg.experiments.questions import load_experiment_questions

QUESTIONS = PROJECT_ROOT / "data" / "processed" / "questions" / "banco-preguntas.csv"
bank = {q.id: q for q in load_experiment_questions(QUESTIONS)}
c2_records = [r for r in records if r["condition"] == "C2"]
wrong_c2_e1 = [
    r for r in c2_records if r["stratum"].startswith("E1") and r["verdict"]["correct"] is False
]
sheet_keys = {(x["pregunta_id"], x["corrida"]) for x in oe4_rows}
omitted = sorted(
    (r["question_id"], r["run"])
    for r in wrong_c2_e1
    if (r["question_id"], str(r["run"])) not in sheet_keys
)
rule_failures = sorted(rule_attributed_failures(c2_records, bank))
assert rule_failures == omitted, "las filas fuera de la hoja son las que atribuye la regla"
print("Respuestas incorrectas de C2 en E1 según los registros:", len(wrong_c2_e1))
print("Filas etiquetadas a mano (hoja):", len(oe4_rows))
print("Atribuidas por la regla y fuera de la hoja:", rule_failures)

Respuestas incorrectas de C2 en E1 según los registros: 43
Filas etiquetadas a mano (hoja): 40
Atribuidas por la regla y fuera de la hoja: [('q025', 0), ('q025', 1), ('q025', 2)]


In [17]:
all_stages = count_stages(["sintesis"] * len(rule_failures), oe4_rows)
all_shares = stage_shares(all_stages)["share"]
n_all = sum(all_stages.values())
assert n_all == len(wrong_c2_e1)
assert all_stages == {"extraccion": 21, "recuperacion": 11, "sintesis": 11}
pd.DataFrame(
    [
        {
            "etapa": stage,
            "casos": all_stages[stage],
            "proporción": pct(all_shares[stage], 1),
            "atribuidos por la regla": len(rule_failures) if stage == "sintesis" else 0,
        }
        for stage in sorted(all_stages, key=lambda s: -all_stages[s])
    ]
).set_index("etapa")

,casos,proporción,atribuidos por la regla
etapa,,,
extraccion,21,"48,8 %",0
recuperacion,11,"25,6 %",0
sintesis,11,"25,6 %",3


**Por qué la regla atribuye esos 3 casos a la síntesis.** La regla (`error_evidence`) cuenta una falla
como de síntesis cuando algún bloque del contexto entregado a C2 contiene todos los términos de
contenido de la respuesta de referencia. La tabla siguiente comprueba esa condición en los 3 casos.

In [18]:
rows_by_run = {
    run: [r for r in c2_records if r["run"] == run]
    for run in sorted({r["run"] for r in c2_records})
}
reason = []
for qid, run in rule_failures:
    run_records = rows_by_run[run]
    context = [x for x in context_rows(run_records, [], bank) if x.question_id == qid]
    (evidence,) = error_evidence(context, run_records, bank)
    reason.append(
        {
            "pregunta": qid,
            "repetición": run,
            "C2 se abstuvo": evidence.abstained,
            "resultado": evidence.outcome,
            "términos de la referencia en el contexto": pct(evidence.reference_coverage, 0),
            "tipo de objeto que los contiene todos": ", ".join(evidence.visible_in),
        }
    )
pd.DataFrame(reason)

,pregunta,repetición,C2 se abstuvo,resultado,términos de la referencia en el contexto,tipo de objeto que los contiene todos
0,q025,0,True,no_answer_in_context,100 %,sources
1,q025,1,True,no_answer_in_context,100 %,sources
2,q025,2,True,no_answer_in_context,100 %,sources


In [19]:
extraction_all = all_shares["extraccion"]
retrieval_all = all_shares["recuperacion"]
synthesis_all = all_shares["sintesis"]
display(
    Markdown(
        f"**Lectura (cifra principal).** De {n_all} respuestas incorrectas de C2 en E1, "
        f"{all_stages['extraccion']} "
        f"({pct(extraction_all, 1)}) se deben a la extracción, {all_stages['recuperacion']} "
        f"({pct(retrieval_all, 1)}) a la recuperación y {all_stages['sintesis']} "
        f"({pct(synthesis_all, 1)}) a la "
        f"síntesis. **La etapa más frecuente es la extracción**: en esos casos el dato no "
        f"llegó a quedar en el "
        f"objeto que la base ofrecía a C2. Los {len(rule_failures)} casos que atribuye la "
        f"regla corresponden a "
        f"{', '.join(sorted({q for q, _ in rule_failures}))} (repeticiones "
        f"{', '.join(str(r) for _, r in rule_failures)}): C2 se abstuvo teniendo la referencia "
        f"completa a la "
        f"vista en un bloque de tipo «{reason[0]['tipo de objeto que los contiene todos']}» "
        f"(la transcripción "
        f"cruda importada a la base), de modo que la regla los cuenta como síntesis. La regla "
        f"compara palabras, "
        f"no significados, y por eso es una pista y no una prueba."
    )
)

**Lectura (cifra principal).** De 43 respuestas incorrectas de C2 en E1, 21 (48,8 %) se deben a la extracción, 11 (25,6 %) a la recuperación y 11 (25,6 %) a la síntesis. **La etapa más frecuente es la extracción**: en esos casos el dato no llegó a quedar en el objeto que la base ofrecía a C2. Los 3 casos que atribuye la regla corresponden a q025 (repeticiones 0, 1, 2): C2 se abstuvo teniendo la referencia completa a la vista en un bloque de tipo «sources» (la transcripción cruda importada a la base), de modo que la regla los cuenta como síntesis. La regla compara palabras, no significados, y por eso es una pista y no una prueba.

## 4.1 Detalle de las filas etiquetadas a mano

Esta subsección es el subconjunto etiquetado por una persona (`gm`), es decir, las filas de
`oe4-etapa-e1.csv`. No incluye los 3 casos que la regla atribuye sola, y por eso sus proporciones
(sobre 40) difieren de las de la cifra principal (sobre 43). Se recalcula el resumen desde la hoja y se
comprueba que coincide con `oe4-resumen.json`.

In [20]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    recomputed_oe4 = summarize_oe4_sheet(oe4_rows, condition=oe4["condition"])
for stage, stored_stage in oe4["stages"].items():
    assert recomputed_oe4.stages[stage] == stored_stage["cases"]
    assert abs(recomputed_oe4.shares[stage] - stored_stage["share"]) < 1e-9
assert recomputed_oe4.n_failures == oe4["n_failures"] == len(oe4_rows)
print("El resumen recalculado desde la hoja coincide con oe4-resumen.json.")
pd.DataFrame(
    [
        {"etapa": stage, "casos": v["cases"], "proporción": pct(v["share"], 1)}
        for stage, v in sorted(oe4["stages"].items(), key=lambda kv: -kv[1]["cases"])
    ]
).set_index("etapa")

El resumen recalculado desde la hoja coincide con oe4-resumen.json.


,casos,proporción
etapa,,
extraccion,21,"52,5 %"
recuperacion,11,"27,5 %"
sintesis,8,"20,0 %"


In [21]:
rule = oe4["rule_agreement"]
q019 = [r for r in oe4_rows if r["pregunta_id"] == "q019"]
display(
    Markdown(
        f"**Lectura.** De {oe4['n_failures']} filas etiquetadas por una persona, la etapa es "
        f"extracción en "
        f"{oe4['stages']['extraccion']['cases']} "
        f"({pct(oe4['stages']['extraccion']['share'], 1)}), recuperación "
        f"en {oe4['stages']['recuperacion']['cases']} "
        f"({pct(oe4['stages']['recuperacion']['share'], 1)}) y "
        f"síntesis en {oe4['stages']['sintesis']['cases']} "
        f"({pct(oe4['stages']['sintesis']['share'], 1)}). La "
        f"pista automática de la hoja solo sugiere etapa en {rule['with_suggestion']} filas y "
        f"en todas coincide "
        f"con la persona ({rule['agree']} de {rule['with_suggestion']}); las otras "
        f"{rule['without_suggestion']} dependen del juicio humano.\n\n"
        f"**Ajuste de q019.** La pregunta q019 ({len(q019)} filas, repeticiones 0 a 2) quedó "
        f"primero como "
        f"síntesis (el objeto recuperado describía el mecanismo y C2 lo reprodujo). `jss` la "
        f"reetiquetó como "
        f"**extracción** porque el detalle de la referencia sí está en la transcripción de la "
        f"serie y falta en "
        f"el objeto recuperado: C2 reprodujo fielmente lo que recuperó y el dato se perdió al "
        f"extraer. La nota "
        f"de esas filas en la hoja registra el cambio y remite a este notebook; el resumen se "
        f"rehízo después "
        f"de él."
    )
)

**Lectura.** De 40 filas etiquetadas por una persona, la etapa es extracción en 21 (52,5 %), recuperación en 11 (27,5 %) y síntesis en 8 (20,0 %). La pista automática de la hoja solo sugiere etapa en 6 filas y en todas coincide con la persona (6 de 6); las otras 34 dependen del juicio humano.

**Ajuste de q019.** La pregunta q019 (3 filas, repeticiones 0 a 2) quedó primero como síntesis (el objeto recuperado describía el mecanismo y C2 lo reprodujo). `jss` la reetiquetó como **extracción** porque el detalle de la referencia sí está en la transcripción de la serie y falta en el objeto recuperado: C2 reprodujo fielmente lo que recuperó y el dato se perdió al extraer. La nota de esas filas en la hoja registra el cambio y remite a este notebook; el resumen se rehízo después de él.

---

# 5. Citas y abstención

**Pregunta.** ¿Cita cada condición evidencia correcta y se abstiene cuando debe?

**Qué se muestra.** La corrección y la precisión de la cita (regla del ADR 0009) con sus intervalos,
tomadas de `analisis.json`; el recuento de citas incorrectas desde `respuestas.jsonl`; y la tasa de
abstención por condición y estrato. **C0 no cita** (no recibe contexto), de modo que sus métricas de
citación y de evidencia no aplican y se informan como no disponibles. En E4 no hay respuesta en el corpus,
así que abstenerse es la conducta esperada; en E1 a E3, una abstención es una respuesta que no se da.

In [22]:
cell_of = {(c["condition"], c["stratum"]): c for c in analysis["cells"]}
cit_rows = []
for condition in ("C1", "C2", "C3"):
    for stratum in ("E1", "E2", "E3", "ALL"):
        metrics = cell_of[(condition, stratum)]["metrics"]
        cit_rows.append(
            {
                "condición": condition,
                "estrato": stratum,
                "cita correcta": ci(metrics["citation_correct"]),
                "precisión de la cita": ci(metrics["citation_precision"]),
            }
        )
pd.DataFrame(cit_rows).pivot(
    index="condición", columns="estrato", values=["cita correcta", "precisión de la cita"]
)

cita correcta                                                          precisión de la cita                                                         
estrato                  ALL                 E1                 E2                 E3                  ALL                 E1                 E2                 E3
condición                                                                                                                                                          
C1         0,70 [0,59; 0,81]  0,69 [0,52; 0,85]  0,67 [0,46; 0,83]  0,75 [0,58; 0,92]    0,59 [0,50; 0,69]  0,66 [0,49; 0,83]  0,53 [0,36; 0,69]  0,59 [0,43; 0,75]
C2         0,45 [0,31; 0,60]  0,38 [0,18; 0,60]  0,55 [0,29; 0,79]  0,42 [0,12; 0,75]    0,39 [0,26; 0,53]  0,28 [0,13; 0,46]  0,55 [0,29; 0,79]  0,36 [0,10; 0,68]
C3         0,71 [0,60; 0,81]  0,68 [0,50; 0,85]  0,71 [0,52; 0,90]  0,74 [0,54; 0,91]    0,59 [0,49; 0,69]  0,55 [0,38; 0,71]  0,59 [0,41; 0,76]  0,65 [0,46; 0,82]

In [23]:
width_rows = []
for condition in ("C1", "C2", "C3"):
    cell = cell_of[(condition, "ALL")]
    width = cell["citation_width"]
    width_rows.append(
        {
            "condición": condition,
            "respuestas con cita evaluable": cell["metrics"]["citation_correct"]["n_records"],
            "actos citados (mediana)": es(width["cited_acts_median"], 0),
            "cobertura media de actos": es(width["mean_act_coverage"], 2),
            "citas a la transcripción completa": width["source_citations"],
        }
    )
pd.DataFrame(width_rows).set_index("condición")

,respuestas con cita evaluable,actos citados (mediana),cobertura media de actos,citas a la transcripción completa
condición,,,,
C1,219,53,"0,34",0
C2,106,7,"0,18",95
C3,189,5,"0,24",79


In [24]:
absten = []
for condition in CONDITIONS:
    sub = [r for r in records if r["condition"] == condition]
    summary = {s.stratum: s for s in stratum_summary(sub)}
    absten.append(
        {"condición": condition}
        | {
            s: f"{summary[s].abstained}/{summary[s].n} ({pct(summary[s].abstention_rate, 0)})"
            for s in ("E1", "E2", "E3", "E4")
        }
    )
pd.DataFrame(absten).set_index("condición").rename_axis("abstenciones / registros")

,E1,E2,E3,E4
abstenciones / registros,,,,
C0,0/75 (0 %),0/75 (0 %),0/72 (0 %),0/75 (0 %)
C1,0/75 (0 %),3/75 (4 %),0/72 (0 %),48/75 (64 %)
C2,27/75 (36 %),38/75 (51 %),51/72 (71 %),75/75 (100 %)
C3,3/75 (4 %),13/75 (17 %),17/72 (24 %),75/75 (100 %)


In [25]:
sub_summary = {
    c: {s.stratum: s for s in stratum_summary([r for r in records if r["condition"] == c])}
    for c in CONDITIONS
}
citation_all = {c: cell_of[(c, "ALL")] for c in ("C1", "C2", "C3")}
display(
    Markdown(
        "**Lectura.** C0 no abstiene en ningún estrato "
        f"({sum(s.abstained for s in sub_summary['C0'].values())} abstenciones) y no cita. La "
        f"cita correcta, "
        "en el total, es "
        + ", ".join(
            f"{c} {ci(citation_all[c]['metrics']['citation_correct'])}" for c in citation_all
        )
        + ": C2 queda por debajo de C1 y C3, que se parecen entre sí. La tabla de anchura "
        "muestra que C2 "
        "tiene menos respuestas con cita evaluable y que parte de sus citas apunta a la "
        "transcripción "
        f"completa de la reunión ({citation_all['C2']['citation_width']['source_citations']} "
        f"citas, "
        f"frente a {citation_all['C3']['citation_width']['source_citations']} en C3 y "
        f"{citation_all['C1']['citation_width']['source_citations']} en C1), que no cuenta "
        f"como evidencia. "
        "C2 se abstiene mucho más que las demás en E1 a E3 ("
        + ", ".join(
            f"{s}: {pct(sub_summary['C2'][s].abstention_rate, 0)}" for s in ("E1", "E2", "E3")
        )
        + "), lo que es coherente con una base a la que le falta el dato (sección 4). En E4, donde "
        f"abstenerse es lo esperado, C2 y C3 se abstienen en "
        f"{pct(sub_summary['C2']['E4'].abstention_rate, 0)} "
        f"de los registros y C1 en {pct(sub_summary['C1']['E4'].abstention_rate, 0)}. La "
        f"precisión y la "
        "cobertura de actos dependen de cuán ancha es la cita, así que una cita correcta no "
        "equivale a "
        "una cita precisa."
    )
)

**Lectura.** C0 no abstiene en ningún estrato (0 abstenciones) y no cita. La cita correcta, en el total, es C1 0,70 [0,59; 0,81], C2 0,45 [0,31; 0,60], C3 0,71 [0,60; 0,81]: C2 queda por debajo de C1 y C3, que se parecen entre sí. La tabla de anchura muestra que C2 tiene menos respuestas con cita evaluable y que parte de sus citas apunta a la transcripción completa de la reunión (95 citas, frente a 79 en C3 y 0 en C1), que no cuenta como evidencia. C2 se abstiene mucho más que las demás en E1 a E3 (E1: 36 %, E2: 51 %, E3: 71 %), lo que es coherente con una base a la que le falta el dato (sección 4). En E4, donde abstenerse es lo esperado, C2 y C3 se abstienen en 100 % de los registros y C1 en 64 %. La precisión y la cobertura de actos dependen de cuán ancha es la cita, así que una cita correcta no equivale a una cita precisa.

---

# 6. Lectura y límites

**Lectura.**

In [26]:
e3 = {
    c["condition"]: c["metrics"]["correct"]["point"]
    for c in analysis["cells"]
    if c["stratum"] == "E3"
}
display(
    Markdown(
        f"- **Dónde falla C2.** En las preguntas de hecho puntual (E1), la extracción es la "
        f"etapa más frecuente ({pct(extraction_all, 1)} de {n_all} fallos). La recuperación "
        f"representa {pct(retrieval_all, 1)} y la síntesis {pct(synthesis_all, 1)}.\n"
        f"- **C3 frente a C2.** Al sustituir las decisiones y sus enlaces por los de la "
        f"referencia bajo la misma consulta, la corrección en E3 pasa de "
        f"{es(e3['C2'])} a {es(e3['C3'])}, una brecha de {es(c3_minus_c2['E3'])}. "
        f"El contraste mide el aporte conjunto de ese cambio en la base.\n"
        f"- **D16 y H2.** El intervalo de la diferencia C2 - C1 en E3 "
        f"({es(d16['difference_lower'])} a {es(d16['difference_upper'])}) queda por debajo de 0. "
        f"La dirección es contraria a la esperada, aunque el programa use el rótulo "
        f"«{d16['criterion']}». El intervalo de H2 incluye 0 y deja abierta la diferencia "
        f"entre las brechas de E3 y E1.\n"
        f"- **Validez del juez.** El acuerdo global para `correct` supera el umbral "
        f"(kappa {es(correct['kappa'], 2)}). Para `evidence_supported` queda por debajo. "
        f"La muestra carece de casos positivos para evaluar la detección de respuestas "
        f"desactualizadas en H4."
    )
)

- **Dónde falla C2.** En las preguntas de hecho puntual (E1), la extracción es la etapa más frecuente (48,8 % de 43 fallos). La recuperación representa 25,6 % y la síntesis 25,6 %.
- **C3 frente a C2.** Al sustituir las decisiones y sus enlaces por los de la referencia bajo la misma consulta, la corrección en E3 pasa de 0,17 a 0,71, una brecha de 0,54. El contraste mide el aporte conjunto de ese cambio en la base.
- **D16 y H2.** El intervalo de la diferencia C2 - C1 en E3 (-0,61 a -0,14) queda por debajo de 0. La dirección es contraria a la esperada, aunque el programa use el rótulo «inconclusive». El intervalo de H2 incluye 0 y deja abierta la diferencia entre las brechas de E3 y E1.
- **Validez del juez.** El acuerdo global para `correct` supera el umbral (kappa 0,81). Para `evidence_supported` queda por debajo. La muestra carece de casos positivos para evaluar la detección de respuestas desactualizadas en H4.

**Alcance de los resultados.**

- **C3 como control.** Las decisiones y los enlaces de C3 provienen de la referencia que también
  sirvió para redactar el banco de preguntas. Así podemos medir qué ocurre al disponer de ese
  contenido bajo la misma consulta de OpenKOS. El 0,80 total corresponde a ese escenario y el
  resto de los objetos de la base sigue siendo de extracción automática (ADR 0013).
- **C3 frente a C1.** El contraste no se preregistró. Sus tasas permiten una comparación
  descriptiva dentro de esta corrida.
- **Extracción.** El diseño y el modelo local (`qwen3:8b`) se probaron juntos. La corrida permite
  localizar en qué etapa se pierde el dato y deja abierta la contribución de cada componente.
- **Verificación del juez.** El acuerdo por condición es exploratorio, con 60 respuestas por
  condición. El menor acuerdo en C0 aconseja cautela al comparar con la referencia sin contexto.
  D16 y H2 comparan condiciones que no incluyen C0. El desacuerdo
  de q015, corrida 2, muestra que el acuerdo global admite errores individuales.
- **Tamaño de la muestra.** Una corrida formal con un generador local pequeño y 99 preguntas:
  25 en E1, 25 en E2, 24 en E3 y 25 en E4. Las tres repeticiones se promedian por pregunta.